# 01 - Exploratory Data Analysis

**Goal:** understand the data before cleaning and training, and confirm the open values in `config.yaml`.

**Requirements:** FR-1, FR-2, FR-3 · **Closes:** OP-1 (department list), `min_text_chars`, `max_length`

## 1. Load the configuration and the raw data

In [10]:

from pathlib import Path   # to build file paths that work on Windows and Linux

import pandas as pd        # tables (DataFrames)
import yaml                # to read config.yaml

print("pandas version:", pd.__version__)

pandas version: 3.1.1


In [11]:
# ===== Cell 1.2: Find the project root =====
# The notebook runs inside the "notebooks" folder, but config.yaml is in the project root.
# So we start from the current folder and go up one folder at a time
# until we find a folder that contains config.yaml.

search_folder = Path.cwd()   # the folder the notebook is running from
project_root = None          # will hold the answer when we find it

while True:
    config_candidate = search_folder / "config.yaml"

    if config_candidate.exists():
        project_root = search_folder
        break

    parent_folder = search_folder.parent

    if parent_folder == search_folder:   # we reached the top of the drive (C:\) and found nothing
        break

    search_folder = parent_folder

if project_root is None:
    raise FileNotFoundError("config.yaml was not found in this folder or in any folder above it.")

print("Project root:", project_root)

Project root: c:\Users\mohme\projedcts\NHA-5-168


In [12]:
# ===== Cell 1.3: Read config.yaml =====

config_path = project_root / "config.yaml"

with open(config_path, "r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

print("Config sections:", list(config.keys()))
print("Random seed:", config["project"]["random_seed"])
print("Raw data path:", config["data"]["raw_path"])

Config sections: ['project', 'data', 'baseline', 'transformer', 'evaluation', 'inference', 'mlflow', 'api', 'logging', 'aws']
Random seed: 42
Raw data path: data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv


In [13]:
# ===== Cell 1.4: Load the raw CSV =====
# All 16 columns are kept for now: this is exploration, dropping columns happens later in preprocess.py.
# Empty cells stay as NaN so that section 3 can count them.

raw_data_path = project_root / config["data"]["raw_path"]
print("Reading:", raw_data_path)

raw_tickets = pd.read_csv(raw_data_path, encoding="utf-8")

number_of_rows = raw_tickets.shape[0]
number_of_columns = raw_tickets.shape[1]
print("Rows:", number_of_rows, "(expected 28,587)")
print("Columns:", number_of_columns, "(expected 16)")

# Show the first rows to check that long e-mails (with commas and new lines) were read into the right columns
raw_tickets.head(3)

Reading: c:\Users\mohme\projedcts\NHA-5-168\data\raw\aa_dataset-tickets-multi-lang-5-2-50-version.csv
Rows: 28587 (expected 28,587)
Columns: 16 (expected 16)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN


In [14]:
# ===== Cell 1.5: Column profile - name, type, missing, empty text, unique values =====
# One row per column of raw_tickets.

total_rows = len(raw_tickets)
profile_rows = []   # a list of dictionaries, one dictionary per column

for column_name in raw_tickets.columns:
    column_values = raw_tickets[column_name]

    # 1) data type as pandas sees it (in pandas 3, text columns show as "str")
    data_type = str(column_values.dtype)

    # 2) missing values (NaN): the cell was empty in the CSV
    missing_count = int(column_values.isna().sum())
    missing_percent = round(100 * missing_count / total_rows, 2)

    # 3) empty text: the cell has a value, but it is only spaces or new lines
    empty_text_count = 0
    for value in column_values:
        if isinstance(value, str):
            if value.strip() == "":
                empty_text_count = empty_text_count + 1

    # 4) number of different values (NaN not counted)
    unique_count = int(column_values.nunique(dropna=True))

    # 5) first real value, as an example (cut to 60 characters)
    example_value = None
    for value in column_values:
        if pd.isna(value):
            continue
        if isinstance(value, str) and value.strip() == "":
            continue
        example_value = str(value)[:60]
        break

    profile_rows.append({
        "column": column_name,
        "dtype": data_type,
        "missing": missing_count,
        "missing_%": missing_percent,
        "empty_text": empty_text_count,
        "unique": unique_count,
        "example": example_value,
    })

column_profile = pd.DataFrame(profile_rows)
column_profile

,column,dtype,missing,missing_%,empty_text,unique,example
0,subject,str,3838,13.43,0,24749,Wesentlicher Sicherheitsvorfall
1,body,str,0,0.00,0,28587,"Sehr geehrtes Support-Team,\n\nich möchte eine..."
2,answer,str,7,0.02,0,28580,Vielen Dank für die Meldung des kritischen Sic...
3,type,str,0,0.00,0,4,Incident
4,queue,str,0,0.00,0,10,Technical Support
5,priority,str,0,0.00,0,3,high
6,language,str,0,0.00,0,2,de
7,version,int64,0,0.00,0,3,51
8,tag_1,str,0,0.00,0,116,Security
9,tag_2,str,13,0.05,0,256,Outage


In [15]:
# ===== Cell 1.6: Duplicate tickets =====
# We check four levels, from strict to loose.

text_columns = config["data"]["text_fields"]                    # ["subject", "body"]
department_column = config["data"]["targets"]["department"]     # "queue"
type_column = config["data"]["targets"]["type"]                 # "type"

# Level 1: rows identical in ALL 16 columns
full_row_duplicates = int(raw_tickets.duplicated().sum())

# Level 2: same subject AND same body (= the same model input)
same_text_duplicates = int(raw_tickets.duplicated(subset=text_columns).sum())

# Level 3: same body only (the subject may differ or be missing)
same_body_duplicates = int(raw_tickets.duplicated(subset=["body"]).sum())

print("Level 1 - identical rows (all columns):", full_row_duplicates)
print("Level 2 - same subject + body:         ", same_text_duplicates)
print("Level 3 - same body only:              ", same_body_duplicates)

# Level 4: same text but DIFFERENT labels -> the data contradicts itself
# Step A: keep only the rows whose text appears more than once
repeated_text_mask = raw_tickets.duplicated(subset=text_columns, keep=False)
repeated_text_rows = raw_tickets[repeated_text_mask]

# Step B: group those rows by their text and check the labels inside each group
groups_with_repeats = 0
groups_with_conflicting_department = 0
groups_with_conflicting_type = 0
rows_in_conflicting_department_groups = 0

for group_key, group_rows in repeated_text_rows.groupby(text_columns, dropna=False):
    groups_with_repeats = groups_with_repeats + 1

    different_departments = group_rows[department_column].nunique(dropna=True)
    if different_departments > 1:
        groups_with_conflicting_department = groups_with_conflicting_department + 1
        rows_in_conflicting_department_groups = rows_in_conflicting_department_groups + len(group_rows)

    different_types = group_rows[type_column].nunique(dropna=True)
    if different_types > 1:
        groups_with_conflicting_type = groups_with_conflicting_type + 1

print()
print("Level 4 - texts that appear more than once:", groups_with_repeats)
print("          of them with different departments:", groups_with_conflicting_department,
      "(rows involved:", rows_in_conflicting_department_groups, ")")
print("          of them with different types:      ", groups_with_conflicting_type)

Level 1 - identical rows (all columns): 0
Level 2 - same subject + body:          0
Level 3 - same body only:               0

Level 4 - texts that appear more than once: 0
          of them with different departments: 0 (rows involved: 0 )
          of them with different types:       0


## 2. Keep English tickets only

In [16]:
# ===== Cell 2.1: Keep English tickets only =====

language_column = "language"
target_language = config["data"]["language"]   # "en"

print("Tickets per language in the raw data:")
language_counts = raw_tickets[language_column].value_counts()
for language_name, count in language_counts.items():
    print("  ", language_name, ":", count)

english_mask = raw_tickets[language_column] == target_language   # True for English rows
english_tickets = raw_tickets[english_mask].copy()                # .copy() = a separate table, safe to change
english_tickets = english_tickets.reset_index(drop=True)           # row numbers 0, 1, 2, ... again

print()
print("English tickets kept:", len(english_tickets), "(expected 16,338)")

Tickets per language in the raw data:
   en : 16338
   de : 12249

English tickets kept: 16338 (expected 16,338)


## 3. Missing and empty values

In [17]:
# ===== Cell 3.1: Missing values in the English tickets =====
# Only columns that have at least one missing value are listed.

english_total = len(english_tickets)
missing_rows = []

for column_name in english_tickets.columns:
    missing_count = int(english_tickets[column_name].isna().sum())
    if missing_count == 0:
        continue
    missing_percent = round(100 * missing_count / english_total, 2)
    missing_rows.append({
        "column": column_name,
        "missing": missing_count,
        "missing_%": missing_percent,
    })

missing_table = pd.DataFrame(missing_rows)
missing_table

,column,missing,missing_%
0,subject,2607,15.96
1,answer,3,0.02
2,tag_2,6,0.04
3,tag_3,69,0.42
4,tag_4,1711,10.47
5,tag_5,7922,48.49
6,tag_6,12968,79.37
7,tag_7,15204,93.06
8,tag_8,16057,98.28


In [18]:
# ===== Cell 3.2: Is a missing subject random, or linked to department / type / priority? =====
# If the missing rate is about the same in every group, the subject is missing at random.
# If one group is much higher, "no subject" carries information about the label.

# Temporary EDA column: True when the ticket has no subject
english_tickets["subject_missing"] = english_tickets["subject"].isna()

overall_missing_rate = round(100 * english_tickets["subject_missing"].mean(), 1)
print("Overall: subject missing in", overall_missing_rate, "% of English tickets")

columns_to_check = [department_column, type_column, "priority"]

for column_name in columns_to_check:
    rows_for_this_column = []

    for category_value, category_rows in english_tickets.groupby(column_name):
        tickets_in_category = len(category_rows)
        missing_in_category = int(category_rows["subject_missing"].sum())
        missing_rate = round(100 * missing_in_category / tickets_in_category, 1)
        difference_from_overall = round(missing_rate - overall_missing_rate, 1)

        rows_for_this_column.append({
            column_name: category_value,
            "tickets": tickets_in_category,
            "no_subject": missing_in_category,
            "no_subject_%": missing_rate,
            "difference_vs_overall": difference_from_overall,
        })

    result_table = pd.DataFrame(rows_for_this_column)
    result_table = result_table.sort_values("no_subject_%", ascending=False)

    print()
    print("By", column_name)
    display(result_table)

Overall: subject missing in 16.0 % of English tickets

By queue


,queue,tickets,no_subject,no_subject_%,difference_vs_overall
2,General Inquiry,236,49,20.8,4.8
7,Sales and Pre-Sales,513,103,20.1,4.1
3,Human Resources,348,63,18.1,2.1
4,IT Support,1942,333,17.1,1.1
6,Returns and Exchanges,820,134,16.3,0.3
9,Technical Support,4737,766,16.2,0.2
0,Billing and Payments,1595,246,15.4,-0.6
1,Customer Service,2410,365,15.1,-0.9
5,Product Support,3073,465,15.1,-0.9
8,Service Outages and Maintenance,664,83,12.5,-3.5



By type


,type,tickets,no_subject,no_subject_%,difference_vs_overall
0,Change,1705,308,18.1,2.1
1,Incident,6571,1040,15.8,-0.2
3,Request,4665,736,15.8,-0.2
2,Problem,3397,523,15.4,-0.6



By priority


,priority,tickets,no_subject,no_subject_%,difference_vs_overall
0,high,6346,1042,16.4,0.4
1,low,3374,542,16.1,0.1
2,medium,6618,1023,15.5,-0.5


## 4. Label distribution: document type (`type`)

## 5. Label distribution: department (`queue`)

## 6. Relationships between columns (type, department, priority, missing subject, tags)

In [19]:
# ===== Cell 6.1: How strongly are the label-like columns related? (Cramér's V) =====
# Cramér's V measures the link between two categorical columns:
#   0   = no relation (knowing one tells nothing about the other)
#   1   = full relation (one column decides the other)
# It is built on the chi-square test: compare the real counts with the counts
# we would expect if the two columns were independent.

def cramers_v(first_column, second_column):
    """Cramér's V between two categorical pandas Series of the same length."""
    contingency = pd.crosstab(first_column, second_column)   # real counts, one cell per pair of values

    total = contingency.values.sum()
    row_totals = contingency.sum(axis=1)
    column_totals = contingency.sum(axis=0)

    chi_square = 0.0
    for row_label in contingency.index:
        for column_label in contingency.columns:
            observed = contingency.loc[row_label, column_label]
            expected = row_totals[row_label] * column_totals[column_label] / total   # count if independent
            chi_square = chi_square + (observed - expected) ** 2 / expected

    number_of_row_values = contingency.shape[0]
    number_of_column_values = contingency.shape[1]
    smaller_dimension = min(number_of_row_values, number_of_column_values) - 1
    if smaller_dimension == 0:
        return 0.0

    return (chi_square / (total * smaller_dimension)) ** 0.5


columns_to_compare = [type_column, department_column, "priority", "subject_missing", "tag_1"]

relation_matrix = pd.DataFrame(index=columns_to_compare, columns=columns_to_compare, dtype=float)

for first_name in columns_to_compare:
    for second_name in columns_to_compare:
        if first_name == second_name:
            relation_matrix.loc[first_name, second_name] = 1.0
        else:
            value = cramers_v(english_tickets[first_name], english_tickets[second_name])
            relation_matrix.loc[first_name, second_name] = round(value, 2)

relation_matrix

,type,queue,priority,subject_missing,tag_1
type,1.00,0.19,0.06,0.02,0.44
queue,0.19,1.00,0.28,0.04,0.34
priority,0.06,0.28,1.00,0.01,0.16
subject_missing,0.02,0.04,0.01,1.00,0.15
tag_1,0.44,0.34,0.16,0.15,1.00


In [20]:
# ===== Cell 6.2: Type vs department - share of each type inside each department =====
# Each row adds up to 100%. If the rows look very different, the type says something about the department.

type_by_department_counts = pd.crosstab(english_tickets[department_column], english_tickets[type_column])

type_by_department_percent = type_by_department_counts.copy().astype(float)
for department_name in type_by_department_counts.index:
    department_total = type_by_department_counts.loc[department_name].sum()
    for type_name in type_by_department_counts.columns:
        count = type_by_department_counts.loc[department_name, type_name]
        type_by_department_percent.loc[department_name, type_name] = round(100 * count / department_total, 1)

type_by_department_percent

type,Change,Incident,Problem,Request
queue,,,,
Billing and Payments,6.0,24.2,21.6,48.3
Customer Service,8.1,25.2,21.6,45.1
General Inquiry,28.4,30.1,20.8,20.8
Human Resources,5.7,52.9,10.3,31.0
IT Support,17.4,35.5,25.2,21.9
Product Support,11.4,45.0,21.7,21.9
Returns and Exchanges,12.8,38.9,19.9,28.4
Sales and Pre-Sales,17.5,20.1,12.1,50.3
Service Outages and Maintenance,22.7,55.6,6.6,15.1


## 7. Text length: characters and DistilBERT tokens

Decide `max_length` and `min_text_chars`.

## 8. Duplicate tickets

## 9. Overlapping departments: read examples

Technical Support, Product Support, Customer Service, IT Support.

## 10. Reference baselines (majority class, most-frequent top-3)

## 11. Findings and decisions for `config.yaml`